In [4]:
import numpy as np
import pandas as pd

In [ ]:
filepath = "../data/raw/customer_churn.csv"  
df1 = pd.read_csv(filepath)
df=df1.copy()

In [14]:
if 'customerID' in df.columns:
    customer_ids = df['customerID'] # ID backup if needed later
    df.drop(columns=['customerID'], inplace=True)

In [15]:
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
df.replace(['None', 'none', 'N/A', 'na'], np.nan, inplace=True)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [16]:
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [17]:
if 'SeniorCitizen' in df.columns:
    df['SeniorCitizen'] = df['SeniorCitizen'].astype(str)

In [ ]:
if 'tenure' in df.columns and 'TotalCharges' in df.columns:
    new_customer_mask = (df['tenure'] == 0) & (df['TotalCharges'].isnull())
    df.loc[new_customer_mask, 'TotalCharges'] = 0.0
    print(f"Imputed {new_customer_mask.sum()} missing TotalCharges for new customers (tenure=0).")
missing_ratio = df.isnull().sum().max() / len(df)
if missing_ratio < 0.02:
    
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

Imputed 11 missing TotalCharges for new customers (tenure=0).


In [19]:
string_cols = df.select_dtypes(include=['object', 'str']).columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()


redundancy_map = {'No internet service': 'No', 'No phone service': 'No'}
df.replace(redundancy_map, inplace=True)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7040,Female,0,Yes,Yes,11,No,No,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes


In [20]:
assert (df['tenure'] >= 0).all(), "Sanity Check Failed: Negative tenure detected!"
assert (df['MonthlyCharges'] >= 0).all(), "Sanity Check Failed: Negative MonthlyCharges detected!"

# Business Logic Verification: Tolerance checking for dynamic calculations
expected_charges_approx = df['tenure'] * df['MonthlyCharges']
# 10% allowance for plan changes/discounts over lifetime
deviations = df[df['TotalCharges'] > (expected_charges_approx * 1.10)]
if not deviations.empty:
    print(f"Warning: {len(deviations)} rows have TotalCharges higher than simple tenure calculation.")

In [21]:
if 'Churn' in df.columns:
    # Target coding binary structure me transform karna
    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0}).astype(int)

In [11]:
assert df.isnull().sum().sum() == 0, "Pipeline Error: Data contains unplanned null values!"

In [28]:
import os
outer_path=r'C:/Users\Administrator/customer-churn-intelligence-platform/data/interim'

interim_path = os.path.join(outer_path, "customer_churn_clean.csv")
df.to_csv(interim_path, index=False)